# CME Futures: Model Analysis

This notebook reads the complete canonical model population produced by `06_linear` through
`10b_stochastic_discount_factor`. Each row retains family, configuration, label, checkpoint, fold
contract, training identity, and prediction identity. No null label is assigned to another
horizon, and checkpoints are not collapsed into a single model label.

IC measures whether a configuration ranks the cross-section correctly on a decision date. It
selects nothing: every row here proceeds to the equal-weight validation backtest in
`13_backtest`, where Sharpe performs selection and the checkpoint is part of what is selected.

What `ic_mean` and `ic_t` are, precisely, because the two readings are easy to confuse. Both are
computed **across folds**: `ic_mean` averages each fold's cross-sectional IC, and `ic_t` divides
that by the *standard error* of the same five numbers, which is their dispersion divided by the
square root of how many are defined - not the dispersion itself. `ic_std` in the table below is
the dispersion, so reproducing `ic_t` from the two columns needs the `sqrt(n_folds_ic)` factor.

The registry also computes the daily-series reading with its HAC standard error, which is the
inferential statistic, but the predictions reader does not surface it, so it is not in the table
below. `ic_n_days` is carried instead: it
counts the validation dates that produced a defined IC, and a configuration whose predictions
collapse to near-constant on some dates has its `ic_mean` measured over fewer of them. Reading
`ic_mean` without `ic_n_days` is how a partial-coverage artifact reads as a leader.

In [1]:
"""Analyze complete CME futures model and causal result catalogs."""

import json

import polars as pl

from case_studies.cme_futures.research_workflow import (
    ALL_LABELS,
    CASE_STUDY,
    MODEL_POPULATION_NAMES,
    official_prediction_catalog,
    open_study,
    product_universe_table,
)
from case_studies.research import CausalResult, require_declared_menu_coverage

In [2]:
EXECUTION_TIER = "canonical"
WORKSPACE: str | None = None

## Complete prediction catalog

The six official population snapshots were created before their model runs. Opening all six and
calling `require_complete` means a failed configuration or checkpoint cannot disappear from this
analysis because another row happened to finish.

In [3]:
if EXECUTION_TIER == "preview" and WORKSPACE is None:
    raise ValueError("preview execution requires WORKSPACE")
study = open_study(execution_tier=EXECUTION_TIER, workspace=WORKSPACE)
universe = product_universe_table()
universe

sector,product,expiry_rule,contract_months
str,str,str,str
"""agriculture""","""ZC""","""business_day_before_15th""","""H,K,N,U,Z"""
"""agriculture""","""ZL""","""business_day_before_15th""","""F,H,K,N,Q,U,V,Z"""
"""agriculture""","""ZM""","""business_day_before_15th""","""F,H,K,N,Q,U,V,Z"""
"""agriculture""","""ZS""","""business_day_before_15th""","""F,H,K,N,Q,U,X"""
"""agriculture""","""ZW""","""business_day_before_15th""","""H,K,N,U,Z"""
…,…,…,…
"""metals""","""SI""","""3rd_last_business_day""","""H,K,N,U,Z"""
"""treasuries""","""ZB""","""last_business_day""","""H,M,U,Z"""
"""treasuries""","""ZF""","""last_business_day""","""H,M,U,Z"""


Canonical analysis reads the six published population snapshots, which is the whole point of
freezing them before their runs. A preview has no published population to read: it is a reduced
re-execution whose rows exist only in its own workspace and which is deliberately excluded from
every official population. It reads its own complete validation predictions instead, and the
comparison against the declared menus below is skipped with them, because a preview fits a named
subset by design and would fail that comparison on every configuration it left out.

In [4]:
if EXECUTION_TIER == "canonical":
    catalog = official_prediction_catalog(study, MODEL_POPULATION_NAMES)
else:
    catalog = (
        study.predictions.table(include_preview=True)
        .filter(
            (pl.col("execution_tier") == "preview")
            & (pl.col("split") == "validation")
            & pl.col("complete")
        )
        .sort("label", "family", "config_name", "checkpoint_kind", "checkpoint_value")
    )
    if catalog.is_empty():
        raise RuntimeError("preview execution registered no complete validation predictions")


def _feature_count(spec_json: str) -> int:
    spec = json.loads(spec_json)
    computation = spec.get("computation", spec)
    return len(computation.get("feature_names") or [])


analysis = catalog.with_columns(
    pl.col("spec_json").map_elements(_feature_count, return_dtype=pl.Int64).alias("feature_count")
).select(
    "family",
    "config_name",
    "label",
    "checkpoint_kind",
    "checkpoint_value",
    "feature_count",
    "ic_mean",
    "ic_t",
    "ic_n_days",
    "n_folds",
    "training_hash",
    "prediction_hash",
)

### Every declared model is here

Each execution notebook checks that it produced everything **it** requested, so none of them can
see a configuration that no notebook requests at all - a menu entry nobody claimed publishes
nothing and every completeness check still passes. This is the one place the families reassemble,
so it is the only place that check can be made. `require_declared_menu_coverage` compares
`(family, label, config_name)` against the training menus and raises on either direction: a
declared model the population omits, or a model in the population that no menu declares.

It returns the rows knowingly excluded, so what this notebook is missing is displayed rather than
taken on trust. `causal_dml` is not in the comparison - it is not a predictive family and the
adapter registry, not a list here, is what decides that.

In [5]:
if EXECUTION_TIER == "canonical":
    excluded = require_declared_menu_coverage(analysis, case_study=CASE_STUDY)
else:
    excluded = analysis.clear()
excluded

family,label,config_name,reason
str,str,str,str


In [6]:
analysis.sort("label", "family", "config_name", "checkpoint_value")

family,config_name,label,checkpoint_kind,checkpoint_value,feature_count,ic_mean,ic_t,ic_n_days,n_folds,training_hash,prediction_hash
str,str,str,str,i64,i64,f64,f64,f64,f64,str,str
"""deep_learning""","""lstm_h64""","""fwd_ret_21d""","""epoch""",5,69,-0.033736,-0.806614,1269.0,5.0,"""bdd1563eec80""","""474588807e3e"""
"""deep_learning""","""lstm_h64""","""fwd_ret_21d""","""epoch""",10,69,-0.013183,-0.272313,1269.0,5.0,"""bdd1563eec80""","""e9e24cbc6247"""
"""deep_learning""","""lstm_h64""","""fwd_ret_21d""","""epoch""",15,69,-0.015797,-0.348159,1269.0,5.0,"""bdd1563eec80""","""f82da9e6e31f"""
"""deep_learning""","""lstm_h64""","""fwd_ret_21d""","""epoch""",20,69,-0.010965,-0.231614,1269.0,5.0,"""bdd1563eec80""","""eb1b8a9f6b0b"""
"""deep_learning""","""lstm_h64""","""fwd_ret_21d""","""epoch""",25,69,-0.011416,-0.245769,1269.0,5.0,"""bdd1563eec80""","""8afc139993db"""
…,…,…,…,…,…,…,…,…,…,…,…
"""tabular_dl""","""tabm_s""","""fwd_ret_5d""","""epoch""",100,69,-0.031573,-2.205986,1285.0,5.0,"""1fe58e2b4a37""","""638ba5ee633b"""
"""tabular_dl""","""tabm_s""","""fwd_ret_5d""","""epoch""",125,69,-0.035334,-2.467665,1285.0,5.0,"""1fe58e2b4a37""","""c3c5fb6d66b6"""
"""tabular_dl""","""tabm_s""","""fwd_ret_5d""","""epoch""",150,69,-0.033916,-2.317324,1285.0,5.0,"""1fe58e2b4a37""","""e487b8628952"""


## Interpretation boundaries

The table compares ranking diagnostics under the declared walk-forward protocol. The backtest
engine supplies portfolio returns, transaction costs, contract sizing, and roll execution before
selection.

Conformal weighting, when used later as an allocator, calibrates chronologically from prior
validation observations. It uses the calibration-window scale and the finite-sample higher order
statistic. It does not fit a scale on the evaluation fold or pool all folds before calibration.

## Causal diagnostics

Double machine learning answers a different question from prediction. The treatment effect is
conditioned on the configured confounders, and HAC uncertainty follows the decision-time order.
A covariance-estimator failure is not relabeled as HAC. The shared runner must return a finite HAC
standard error for a result to be complete.

**The `refutation_p` column below is not evidence that these effects survived a placebo test.**
The refutation permutes contiguous blocks within each product, and the shared runner sizes those
blocks from the label buffer rather than from the treatment: `block_size` is set equal to
`embargo`, so the registered rows carry a 21-period block for `fwd_ret_21d` and a 5-period block
for `fwd_ret_5d`. Neither length is a property of `carry_pct`. Measured on this case study's own
feature panel, `carry_pct` has a lag-1 autocorrelation of 0.943, an AR(1) half-life of 11.8
trading days, and autocorrelation still at 0.44 by lag 21 and 0.17 by lag 63. Blocks of 5 and 21
periods therefore destroy serial dependence that the real treatment has. That narrows the placebo
distribution relative to the true null and pushes the empirical p-value toward zero whether or not
the effect is real, which is what both labels report. Read the DML point estimate and its HAC
standard error. The refutation column is recorded for completeness and carries no evidence here.

In [7]:
causal_rows = []
for label in ALL_LABELS:
    result = CausalResult.one(study, label=label, execution_tier=EXECUTION_TIER)
    if not result.complete:
        raise RuntimeError(f"causal result for {label} is incomplete")
    causal_rows.append({"label": label, "causal_hash": result.hash, **result.metrics})
causal = pl.DataFrame(causal_rows).sort("label")

In [8]:
causal

label,causal_hash,n_obs,dml_effect,dml_se_hac,p_value_hac,naive_effect,confounding_bias_pct,refutation_p,refutation_n_successful,placebo_effects,refutation_class
str,str,i64,f64,f64,f64,f64,f64,f64,i64,list[f64],str
"""fwd_ret_21d""","""a8d63faae334""",73239,-0.016256,0.008739,0.062963,-0.008765,46.084494,0.009901,100,"[0.000774, 0.005316, … 0.003168]","""Passes"""
"""fwd_ret_5d""","""711282381c82""",74057,-0.002298,0.002474,0.352955,-0.002069,9.983893,0.039604,100,"[0.001101, 0.002165, … 0.000665]","""Passes"""


## What proceeds to backtesting

All complete prediction rows proceed. The next notebook passes the selected Polars rows directly
to the shared backtest call, publishes product-keyed decisions, and records contract, roll, price,
and prediction lineage. Validation backtest Sharpe, with the prediction checkpoint included in the
configuration identity, is the selection statistic.